# 6장 2강 : uv를 활용한 의존성 관리
## 학습 목표
- uv add, remove 명령어를 활용하여 외부 오픈소스 패키지를 내 프로젝트에 추가 또는 제거할 수 있다.
- pyproject.toml과 uv.lock 파일이 동기화되며 의존성을 고정하는 현대적 빌드 구조를 설명할 수 있다.

### 1. 오픈소스 생태계와 선언적 의존성 관리 구조
---
#### 1.1 오픈소스 중앙 저장소 PyPI
##### PyPI(Python Package Index) /// PyPI PyPI · The Python Package Index (https://pypi.org/)

- 의존성(Dependency)이란?
- 프로젝트가 원활하게 동작하기 위해 외부 라이브러리에 의존하는 상태를 의미합니다. 예를 들어, 내가 만든 프로그램이 A 패키지를 사용하고, A 패키지가 다시 B 패키지를 기반으로 동작한다면 B 역시 내 프로젝트의 의존성이 됩니다. 이러한 복잡한 연결 고리를 체계적으로 추적하고 관리하지 않으면 버전이 꼬여 프로그램이 먹통이 되는 버그가 발생합니다.

#### 1.2 uv add 명령어를 통한 선언적 의존성 관리
**명령 기반의 선언적 관리**

---

**“uv add 패키지명”** 명령어를 사용하면 개발자가 수동으로 압축 파일을 풀거나 환경 변수를 세팅할 필요가 없습니다. uv는 해당 명령을 인지하는 즉시 PyPI 저장소에서 최적의 패키지 버전을 검색하고, 프로젝트 설정 파일에 이를 명시적으로 선언하여 관리하는 동작 방식을 취합니다.

**pip 방식과의 성능 비교**

---

기존의 **pip** 방식은 패키지를 순차적으로 다운로드하고 의존성 충돌 여부를 뒤늦게 확인하여 속도가 느리고 에러에 취약했습니다. 반면 uv는 러스트(Rust) 기반의 고속 병렬 알고리즘을 가동하여 의존성 그래프를 선제적으로 식별하고 다운로드를 동시에 수행하므로 압도적인 속도와 안정성을 보여줍니다.

### 2. pyproject.toml 동기화와 uv.lock 파일의 역할
---
#### 2.1 pyproject.toml 파일의 의존성 자동 동기화

<aside>

**dependencies 필드**

---

**uv add**를 실행하면 프로젝트 루트 디렉터리에 있는 파이썬 표준 설정 파일인 `pyproject.toml` 내부에 해당 패키지 정보가 자동으로 기입됩니다. 파일 내의 **[project]** 하위 **dependencies** 항목에 내가 추가한 외장 라이브러리와 필요한 최소 버전 조건이 텍스트 형태로 명시됩니다.

**자동 동기화의 이점**

---

개발자가 직접 텍스트를 편집하면서 발생할 수 있는 오타나 누락 실수를 방지합니다. **uv**는 명령어를 실행하는 즉시 명세 파일을 자동으로 업데이트하므로, 프로젝트의 의존성 상태를 항상 최신으로 안전하게 유지할 수 있습니다.

</aside>

#### 2.2 버전 충돌을 방어하는 uv.lock 파일과 uv sync 워크플로우

<aside>

**uv.lock 스냅샷의 역할**

---

`pyproject.toml`이 대략적인 버전 조건(예: **pandas>=2.0**)을 선언한다면, `uv.lock` 파일은 실제로 내 컴퓨터에 다운로드된 패키지의 고유 해시(Hash) 수치와 정확한 버전 번호를 소수점 단위까지 정밀하게 기록해 두는 보안 잠금 파일입니다.

**버전 충돌 방지**

---

여러 명의 개발자가 협업할 때 각자의 컴퓨터에 설치되는 오픈소스 버전이 미세하게 달라 프로그램이 동작하지 않는 '버전 충돌 현상'을 원천 방어합니다. 모든 팀원이 동일한 `uv.lock` 파일을 공유하면 정확히 일치하는 환경이 빌드됩니다.

**uv sync 고속 복원 워크플로우 원리**

---

만약 깃(Git) 등에서 팀원의 코드를 새로 내려받았거나 기존 설정이 유실되었을 때, 터미널에 **uv sync** 명령어를 가동하면 uv는 `uv.lock`에 기록된 완벽한 환경 스냅샷과 현재 내 가상환경을 비교합니다. 누락되거나 변경된 알맹이 패키지만을 초고속으로 추적하여 일괄 동기화를 수행하며, 이를 통해 언제 어디서나 오차 없는 동일한 실행 흐름을 복원해 냅니다.

</aside>

## 3. uv 기반 외장 패키지 추가 및 프로젝트 동기화 구현
---
#### 3.1 uv init 프로젝트 생성 및 외장 패키지 설치

<aside>

**버전 명시 프로젝트 생성**

---

터미널을 열고 uv init 명령어를 실행하여 파이썬 3.13 버전의 새 프로젝트를 생성합니다.

```bash
uv init my_dependency_project --python 3.13
```

**외부 패키지 설치**

---

새롭게 자동 생성된 프로젝트 폴더 공간으로 진입합니다.

```bash
cd my_dependency_project
```

uv add 명령어를 실행하여 pandas와 matplotlib 패키지를 프로젝트 설치합니다.

```bash
uv add pandas matplotlib
```

**uv add pandas matplotlib**이 실행되면 PyPI 서버와의 통신을 통해 두 라이브러리의 최신 안정화 버전을 조회합니다.

</aside>

#### 3.2 pyproject.toml 변경 검증 및 uv sync 복원

<aside>

**잠금 파일(uv.lock) 생성 확인**

---

패키지가 추가되면서 빌드된 `uv.lock` 파일의 의존성 명세 정보와 해시 상태를 확인 합니다.

```toml
version = 1
revision = 3
requires-python = ">=3.13"
resolution-markers = [
    "python_full_version >= '3.14' and sys_platform == 'win32'",
    "python_full_version >= '3.14' and sys_platform == 'emscripten'",
    "python_full_version >= '3.14' and sys_platform != 'emscripten' and sys_platform != 'win32'",
    "python_full_version < '3.14' and sys_platform == 'win32'",
    "python_full_version < '3.14' and sys_platform == 'emscripten'",
    "python_full_version < '3.14' and sys_platform != 'emscripten' and sys_platform != 'win32'",
]

[[package]]
name = "contourpy"
version = "1.3.3"
source = { registry = "https://pypi.org/simple" }
dependencies = [
    { name = "numpy" },
]

...
```

</aside>

**uv sync 복원**

---

.venv 폴더를 삭제하여 가상환경을 제거하고 uv sync 명령어 실행으로 복원 되는지 확인 합니다.

### 4.2 개념 요약 키워드

- **PyPI 저장소:** 전 세계 파이썬 개발자들이 완성해 둔 대규모 오픈소스 외장 라이브러리를 안전하게 공유하고 통합 보관하는 공식 플랫폼입니다.
- **선언적 의존성 관리:** 수동 다운로드 방식 대신 프로젝트 명세 파일에 필요한 외장 패키지를 선언해 두면 uv 엔진이 알아서 탐색 및 주입을 전담하는 고도화된 관리 방식입니다.
- **pyproject.toml 동기화:** **uv add** 명령어 실행 즉시 **dependencies** 필드 구조 내부에 추가된 라이브러리 명칭과 버전 요건이 명세 정보 형태로 자동 기입되는 동작 방식입니다.
- **uv.lock 보안 잠금:** 내 프로젝트 가상환경에 이식된 모든 패키지의 정밀한 버전 번호와 sha256 고유 해시값을 기록하여 개발자 간 버전 충돌을 차단하는 스냅샷 파일입니다.
- **uv sync 워크플로우:** 잠금 명세 정보와 현재 가상환경의 설치 상태를 일괄 비교하여, 훼손되거나 누락된 오픈소스만을 고속 복원하는 환경 일치화 제어 방식입니다.

### 4.3 용어 사전

| 용어 | 영어 표기 | 설명 |
|---|---|---|
| **의존성** | Dependency | 프로젝트를 개발할 때 필요한 모든 기능을 직접 만드는 대신, 외부 개발자들이 검증해 둔 오픈소스 소프트웨어나 라이브러리를 가져와 결합하여 사용하는 상태를 뜻합니다. |
| **PyPI** | PyPI (Python Package Index) | 전 세계 파이썬 개발자들이 제작한 수많은 오픈소스 패키지가 모여 있는 공식 중앙 저장소 플랫폼입니다. uv는 이 저장소와 연동되어 패키지를 자동으로 찾아 다운로드합니다. |
| **선언적 의존성 관리** | Declarative Dependency Management | 컴퓨터에 패키지를 수동으로 직접 다운로드하는 대신, 프로젝트 설정 파일에 필요한 패키지 명칭과 조건을 텍스트로 명시하여 관리 도구가 알아서 환경을 제어하도록 하는 방식입니다. |
| **pyproject.toml** | pyproject.toml | 최신 파이썬 표준 규격을 따르는 프로젝트 통합 설정 파일입니다. 이 파일 내부의 `dependencies` 항목에 프로젝트가 사용하는 외부 패키지 이름이 기록됩니다. |
| **uv.lock** | uv.lock | 실제 설치에 사용되는 외부 라이브러리의 정확한 버전과 관련 정보를 기록해 두는 잠금 파일입니다. 팀원들이 동일한 패키지 버전을 사용하도록 도와 환경 차이로 인한 문제를 줄입니다. |
| **uv add** | uv add | 터미널에서 외부 패키지를 현재 프로젝트에 추가하는 명령어입니다. 실행하면 패키지를 설치하고 `pyproject.toml`과 `uv.lock`을 함께 갱신합니다. |
| **uv sync** | uv sync | `pyproject.toml`과 `uv.lock`을 기준으로 현재 프로젝트의 가상환경을 필요한 상태로 맞춰주는 명령어입니다. 가상환경을 다시 만들거나 팀원의 변경사항을 반영할 때 사용할 수 있습니다. |